# Sistema de Recomendação com Aprendizado por Reforço

Este notebook implementa um sistema de recomendação cooperativo com **cinco agentes DQN paralelos**, treinados usando `gymnasium.Env` e PyTorch puro (sem dependência de bibliotecas de RL externas).

## Arquitetura

```
                  ┌─────────────────── Estado Compartilhado ───────────────────┐
                  │  média( features dos 50 últimos filmes do usuário )         │
                  │  média( features dos 500 últimos filmes recomendados )      │
                  └────────────────────────────────────────────────────────────┘
                         │              │              │              │              │
                         ▼              ▼              ▼              ▼              ▼
                    Agente 1       Agente 2       Agente 3       Agente 4       Agente 5
                     Gênero     Gênero Dir.     Raça Dir.      Região Dir.      Idioma
                   (n_genres)    [M|F|MIX]    [W|NW|MIX]     (9 regiões)     (n_langs)
```

## Arquivos necessários

| Arquivo | Descrição |
|---------|----------|
| `movie_encoding_director.tsv` | Features de filmes + encodings de diversidade do diretor |
| `user_ratings.csv` | Avaliações dos usuários (USERID, MOVIEID, RATING) |
| `recommendations.tsv` | Histórico de recomendações do sistema (opcional) |

In [ ]:
# Dependências: gymnasium (compatível com NumPy 2.x + Python 3.12), PyTorch, pandas
!pip install gymnasium torch numpy pandas --quiet

import gymnasium, torch, numpy, pandas
print(f'gymnasium {gymnasium.__version__}  |  torch {torch.__version__}  |  numpy {numpy.__version__}')

In [ ]:
import os, re, random, math
from collections import deque, defaultdict, Counter
from functools import partial
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

import gymnasium as gym
from gymnasium import spaces

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Dispositivo: {DEVICE}')

## 2. Upload dos Arquivos

Faça upload dos arquivos gerados pelo projeto `rs`.
- **Opção A**: upload direto pelo Colab
- **Opção B**: montar o Google Drive e apontar o caminho

In [ ]:
import os

# ── Opção A: upload direto ─────────────────────────────────────────────────────
from google.colab import files
uploaded = files.upload()   # selecione: movie_encoding_director.tsv, user_ratings.csv
BASE = '/content/'

# ── Opção B: Google Drive (comente o bloco acima e descomente este) ─────────────
# from google.colab import drive
# drive.mount('/content/drive')
# BASE = '/content/drive/MyDrive/rs_data/'

MOVIE_ENC_PATH = os.path.join(BASE, 'movie_encoding_director.tsv')
RATINGS_PATH   = os.path.join(BASE, 'user_ratings.csv')
RECS_PATH      = os.path.join(BASE, 'recommendations.tsv')   # opcional

print('movie_encoding_director.tsv:', os.path.exists(MOVIE_ENC_PATH))
print('user_ratings.csv:           ', os.path.exists(RATINGS_PATH))
print('recommendations.tsv:        ', os.path.exists(RECS_PATH), '(opcional)')

## 3. Carregamento e Exploração dos Dados

O arquivo `movie_encoding_director.tsv` contém:
- `movieid`, `imdbid`, `title`
- `genre_*` — colunas binárias de gênero do filme
- `lang_*` — colunas binárias de idioma do filme
- `dir_gender_*` — encoding de gênero do diretor (ex: Exclusive_Male, Majority_Female…)
- `dir_race_*` — encoding de raça do diretor
- `dir_region_*` — encoding de região de origem do diretor

In [ ]:
print('Carregando movie_encoding_director.tsv ...')
enc_df = pd.read_csv(MOVIE_ENC_PATH, sep='\t', low_memory=False)

# Identifica grupos de colunas automaticamente
genre_cols  = [c for c in enc_df.columns if c.startswith('genre_')]
lang_cols   = [c for c in enc_df.columns if c.startswith('lang_')]
gender_cols = [c for c in enc_df.columns if c.startswith('dir_gender_')]
race_cols   = [c for c in enc_df.columns if c.startswith('dir_race_')]
region_cols = [c for c in enc_df.columns if c.startswith('dir_region_')]

print(f'Filmes       : {len(enc_df):,}')
print(f'Gêneros      : {len(genre_cols)}')
print(f'Idiomas      : {len(lang_cols)}')
print(f'Dir. gênero  : {len(gender_cols)}  → {[c.replace("dir_gender_","") for c in gender_cols]}')
print(f'Dir. raça    : {len(race_cols)}  → {[c.replace("dir_race_","") for c in race_cols]}')
print(f'Dir. região  : {len(region_cols)}  → {[c.replace("dir_region_","") for c in region_cols]}')

print('\nCarregando user_ratings.csv ...')
ratings_df = pd.read_csv(RATINGS_PATH)
ratings_df.columns = ratings_df.columns.str.upper()
print(f'Avaliações   : {len(ratings_df):,}')
print(f'Usuários     : {ratings_df["USERID"].nunique():,}')
print(f'Filmes avaliados: {ratings_df["MOVIEID"].nunique():,}')
print(f'Rating médio : {ratings_df["RATING"].mean():.2f}')

# Recomendações anteriores (opcional)
if os.path.exists(RECS_PATH):
    recs_df = pd.read_csv(RECS_PATH, sep='\t')
    recs_df.columns = recs_df.columns.str.upper()
    print(f'\nRecomendações: {len(recs_df):,}')
else:
    recs_df = None
    print('\nArquivo recommendations.tsv não encontrado — histórico de sistema vazio.')

In [ ]:
# Distribuição dos dados — visualização rápida
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
fig.suptitle('Distribuição de filmes por diversidade do diretor', fontsize=13)

for ax, cols, title in zip(axes,
    [gender_cols, race_cols, region_cols],
    ['Gênero do Diretor', 'Raça do Diretor', 'Região de Origem']):
    counts = enc_df[cols].sum().sort_values(ascending=False)
    counts.index = counts.index.str.split('_', n=2).str[-1]  # remove prefixo
    counts.plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(title)
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

## 4. Mapeamento de Grupos de Diversidade

Os Agentes 2, 3 e 4 trabalham com **grupos agregados** derivados das colunas de encoding:

| Agente | Grupos | Colunas agregadas |
|--------|--------|-----------------|
| 2 — Gênero do Dir. | MALE, FEMALE, MIX | `dir_gender_*` agrupadas por padrão no nome |
| 3 — Raça do Dir. | WHITE, NO_WHITE, MIX | `dir_race_*` agrupadas por padrão no nome |
| 4 — Região do Dir. | 9 regiões | `dir_region_*` diretas (1 coluna = 1 região) |

O agrupamento é feito por **correspondência de padrão no nome da coluna**, tornando o código robusto a variações de nomenclatura.

In [ ]:
# ── Agente 2: grupos de gênero do diretor ─────────────────────────────────────
GENDER_GROUPS = ['MALE', 'FEMALE', 'MIX']

def _match_gender(col: str) -> Optional[int]:
    c = col.replace('dir_gender_', '').lower()
    if 'female' in c:                        return 1   # FEMALE (antes de male!)
    if 'male' in c:                          return 0   # MALE
    if 'unknown' in c or 'equal' in c:       return 2   # MIX
    return None

GENDER_COL_GROUP = {c: _match_gender(c) for c in gender_cols}
print('Mapeamento de gênero do diretor:')
for col, grp in GENDER_COL_GROUP.items():
    label = GENDER_GROUPS[grp] if grp is not None else '?'
    print(f'  {col:<45} → {label}')

# ── Agente 3: grupos de raça do diretor ──────────────────────────────────────
RACE_GROUPS = ['WHITE', 'NO_WHITE', 'MIX']

def _match_race(col: str) -> Optional[int]:
    c = col.replace('dir_race_', '').lower()
    if 'no_white' in c or 'nonwhite' in c:   return 1   # NO_WHITE (antes de white!)
    if 'white' in c:                          return 0   # WHITE
    if 'unknown' in c or 'mixed' in c:        return 2   # MIX
    return None

RACE_COL_GROUP = {c: _match_race(c) for c in race_cols}
print('\nMapeamento de raça do diretor:')
for col, grp in RACE_COL_GROUP.items():
    label = RACE_GROUPS[grp] if grp is not None else '?'
    print(f'  {col:<45} → {label}')

# ── Agente 4: região do diretor (direto das colunas) ─────────────────────────
REGION_NAMES = [c.replace('dir_region_', '').replace('_', ' ').title() for c in region_cols]
print(f'\nRegiões ({len(region_cols)}): {REGION_NAMES}')

## 5. Embeddings

Variáveis textuais e categóricas são transformadas em **vetores densos** antes do treinamento:

| Feature | Tipo | Dimensão | Técnica |
|---------|------|----------|---------|
| Título | texto | 32 | `nn.EmbeddingBag` sobre tokens de palavras |
| Gênero do filme | multi-hot | 16 | Projeção linear treinável |
| Idioma do filme | multi-hot | 8 | Projeção linear treinável |
| Dir. gênero/raça/região | multi-hot | direto | Sem projeção (já numérico) |

As projeções são inicializadas aleatoriamente e pré-computadas uma vez. Em produção,
poderiam ser ajustadas conjuntamente com o treinamento dos agentes.

In [ ]:
# ── Vocabulário para título ────────────────────────────────────────────────────
def _tokenize(text: str) -> list[str]:
    return re.sub(r'[^a-z0-9 ]', '', str(text).lower()).split()

word_counter = Counter()
for title in enc_df['title'].fillna(''):
    word_counter.update(_tokenize(title))

VOCAB = ['<PAD>', '<UNK>'] + [w for w, _ in word_counter.most_common(8000)]
WORD2IDX = {w: i for i, w in enumerate(VOCAB)}
print(f'Vocabulário: {len(VOCAB)} tokens')

# ── Dimensões dos embeddings ──────────────────────────────────────────────────
TITLE_DIM  = 32
GENRE_DIM  = 16
LANG_DIM   = 8

N_GENRES  = len(genre_cols)
N_LANGS   = len(lang_cols)
N_GENDER  = len(gender_cols)
N_RACE    = len(race_cols)
N_REGION  = len(region_cols)

# Dimensão total do vetor de features de um filme
MOVIE_FEAT_DIM = TITLE_DIM + GENRE_DIM + LANG_DIM + N_GENDER + N_RACE + N_REGION
# Dimensão do estado (obs) = 2 × MOVIE_FEAT_DIM (user_hist + sys_hist)
OBS_DIM = 2 * MOVIE_FEAT_DIM
print(f'Feature dim por filme : {MOVIE_FEAT_DIM}')
print(f'Observation dim       : {OBS_DIM}')


class MovieEmbedder(nn.Module):
    """
    Converte features brutas de um filme em um vetor denso (MOVIE_FEAT_DIM).

    Componentes:
      - title_emb   : EmbeddingBag sobre tokens de palavras do título
      - genre_proj  : projeção linear do multi-hot de gêneros
      - lang_proj   : projeção linear do multi-hot de idiomas
      - dir_*       : vetores de encoding do diretor usados diretamente
    """
    def __init__(self):
        super().__init__()
        self.title_emb  = nn.EmbeddingBag(len(VOCAB), TITLE_DIM, mode='mean', padding_idx=0)
        self.genre_proj = nn.Linear(N_GENRES, GENRE_DIM)
        self.lang_proj  = nn.Linear(N_LANGS,  LANG_DIM)

    def forward(self, title_tokens, genre_vec, lang_vec, dir_vec):
        """title_tokens: (B, T) | genre_vec: (B, N_GENRES) | ..."""
        t_emb = self.title_emb(title_tokens)          # (B, TITLE_DIM)
        g_emb = F.relu(self.genre_proj(genre_vec))    # (B, GENRE_DIM)
        l_emb = F.relu(self.lang_proj(lang_vec))      # (B, LANG_DIM)
        return torch.cat([t_emb, g_emb, l_emb, dir_vec], dim=-1)  # (B, MOVIE_FEAT_DIM)


embedder = MovieEmbedder()
print(f'\nMovieEmbedder criado. Parâmetros: {sum(p.numel() for p in embedder.parameters()):,}')

In [ ]:
def _gender_group_vec(gender_row: np.ndarray) -> np.ndarray:
    """Agrega colunas de gênero do diretor em 3 grupos: [MALE, FEMALE, MIX]."""
    vec = np.zeros(3, dtype=np.float32)
    for i, col in enumerate(gender_cols):
        grp = GENDER_COL_GROUP.get(col)
        if grp is not None:
            vec[grp] += float(gender_row[i])
    return vec

def _race_group_vec(race_row: np.ndarray) -> np.ndarray:
    """Agrega colunas de raça do diretor em 3 grupos: [WHITE, NO_WHITE, MIX]."""
    vec = np.zeros(3, dtype=np.float32)
    for i, col in enumerate(race_cols):
        grp = RACE_COL_GROUP.get(col)
        if grp is not None:
            vec[grp] += float(race_row[i])
    return vec


def build_movie_features(df: pd.DataFrame) -> dict:
    """
    Pré-computa vetores de features para todos os filmes.
    Retorna {movieid: dict_de_features}.
    """
    embedder.eval()
    features = {}

    for _, row in df.iterrows():
        try:
            mid = int(row['movieid'])
        except (ValueError, TypeError):
            continue

        # Tokeniza o título
        words  = _tokenize(str(row.get('title', '')))
        tokens = [WORD2IDX.get(w, 1) for w in words] or [0]   # 1 = <UNK>

        # Vetores brutos das features
        genre_v  = row[genre_cols].values.astype(np.float32)
        lang_v   = row[lang_cols].values.astype(np.float32)
        gender_v = row[gender_cols].values.astype(np.float32)
        race_v   = row[race_cols].values.astype(np.float32)
        region_v = row[region_cols].values.astype(np.float32)
        dir_v    = np.concatenate([gender_v, race_v, region_v])

        # Embedding denso
        with torch.no_grad():
            feat_vec = embedder(
                torch.tensor([tokens], dtype=torch.long),
                torch.tensor([genre_v]),
                torch.tensor([lang_v]),
                torch.tensor([dir_v]),
            ).squeeze(0).numpy()

        features[mid] = {
            'vec':          feat_vec,       # vetor denso (MOVIE_FEAT_DIM,)
            'genres':       genre_v,        # multi-hot gêneros
            'langs':        lang_v,         # multi-hot idiomas
            'gender_raw':   gender_v,       # encoding bruto gênero dir.
            'race_raw':     race_v,         # encoding bruto raça dir.
            'region':       region_v,       # encoding região dir. (9 dims)
            'gender_group': _gender_group_vec(gender_v),  # 3 grupos
            'race_group':   _race_group_vec(race_v),      # 3 grupos
        }

    return features


print('Pré-computando features dos filmes (pode levar ~1-2 min)...')
movie_features = build_movie_features(enc_df)
all_movie_ids  = list(movie_features.keys())
print(f'Pronto! {len(movie_features):,} filmes com vetores de {MOVIE_FEAT_DIM} dimensões.')

## 6. Carregador de Dados

O `DataLoader` organiza as estruturas de acesso rápido:
- `ratings` — dicionário `{user_id → {movie_id → rating}}`
- `rec_history` — dicionário `{user_id → [movie_ids recomendados]}`
- Split 80/20 entre usuários de treino e teste

In [ ]:
def build_ratings(df: pd.DataFrame) -> dict:
    """Constrói {user_id: {movie_id: rating}} a partir do DataFrame."""
    out: dict[int, dict[int, float]] = defaultdict(dict)
    for _, row in df.iterrows():
        try:
            uid = int(row['USERID'])
            mid = int(row['MOVIEID'])
            rat = float(row['RATING'])
        except (ValueError, TypeError, KeyError):
            continue
        out[uid][mid] = rat
    return dict(out)


def build_rec_history(df: Optional[pd.DataFrame]) -> dict:
    """Constrói {user_id: [movie_ids recomendados]} a partir do DataFrame de recomendações."""
    if df is None:
        return {}
    out: dict[int, list[int]] = defaultdict(list)
    mid_col = next((c for c in df.columns if 'MOVIE' in c or 'ITEM' in c), None)
    if mid_col is None:
        return {}
    for _, row in df.iterrows():
        try:
            uid = int(row['USERID'])
            mid = int(row[mid_col])
            out[uid].append(mid)
        except (ValueError, TypeError):
            continue
    return dict(out)


ratings     = build_ratings(ratings_df)
rec_history = build_rec_history(recs_df)
all_users   = list(ratings.keys())

# Split 80/20
random.shuffle(all_users)
split = int(len(all_users) * 0.8)
train_users = all_users[:split]
test_users  = all_users[split:]

print(f'Usuários totais : {len(all_users):,}')
print(f'  Treino        : {len(train_users):,}')
print(f'  Teste         : {len(test_users):,}')

## 7. Ambiente `gymnasium.Env`

### Estado (observação)

```
obs = [ mean(features dos 50 últimos filmes do usuário)  ]  ← preferências do usuário
    + [ mean(features dos 500 últimos filmes recomendados) ]  ← viés do sistema
```

### Ação

Os cinco agentes fornecem simultaneamente seus vetores de probabilidade:

```python
actions = {
    'genre':    array(n_genres,)   # Agente 1
    'gender':   array(3,)          # Agente 2 — [MALE, FEMALE, MIX]
    'race':     array(3,)          # Agente 3 — [WHITE, NO_WHITE, MIX]
    'region':   array(n_regions,)  # Agente 4
    'language': array(n_langs,)    # Agente 5
}
```

### Episódio

Cada episódio = uma sessão de recomendação com N passos para um usuário sorteado.
Cada passo = uma recomendação de filme.

In [ ]:
def _softmax(x: np.ndarray) -> np.ndarray:
    e = np.exp(x - x.max())
    return e / e.sum()


class MovieRecommendationEnv(gym.Env):
    """
    Ambiente de recomendação cooperativo com 5 agentes paralelos.

    Cada passo:
      1. Os 5 agentes fornecem distribuições de probabilidade.
      2. O ambiente seleciona um filme compatível.
      3. Retorna 5 recompensas independentes (uma por agente).
      4. Atualiza o histórico do sistema.
    """

    HIST_USER   = 50    # últimos N filmes vistos pelo usuário
    HIST_SYS    = 500   # últimos N filmes recomendados pelo sistema
    MAX_STEPS   = 10    # passos (recomendações) por episódio
    MAX_CAND    = 500   # máximo de candidatos avaliados por passo

    def __init__(self, features: dict, ratings: dict, rec_history: dict, user_ids: list):
        super().__init__()
        self.features     = features
        self.ratings      = ratings
        self.rec_history  = rec_history
        self.user_ids     = user_ids

        # Espaço de observação: vetor contínuo (user_hist + sys_hist médias)
        self.observation_space = spaces.Box(
            -np.inf, np.inf, shape=(OBS_DIM,), dtype=np.float32
        )

        # Espaço de ação: dicionário com probabilidades de cada agente
        self.action_space = spaces.Dict({
            'genre':    spaces.Box(0., 1., shape=(N_GENRES,),  dtype=np.float32),
            'gender':   spaces.Box(0., 1., shape=(3,),          dtype=np.float32),
            'race':     spaces.Box(0., 1., shape=(3,),          dtype=np.float32),
            'region':   spaces.Box(0., 1., shape=(N_REGION,),   dtype=np.float32),
            'language': spaces.Box(0., 1., shape=(N_LANGS,),    dtype=np.float32),
        })

        self._user_id   = None
        self._user_hist = deque(maxlen=self.HIST_USER)
        self._sys_hist  = deque(maxlen=self.HIST_SYS)
        self._seen      = set()
        self._step_n    = 0

    # ── Observação ────────────────────────────────────────────────────────────

    def _mean_feat(self, movie_ids) -> np.ndarray:
        """Média dos vetores densos de uma lista de filmes."""
        vecs = [self.features[m]['vec'] for m in movie_ids if m in self.features]
        return np.mean(vecs, axis=0).astype(np.float32) if vecs \
               else np.zeros(MOVIE_FEAT_DIM, dtype=np.float32)

    def _obs(self) -> np.ndarray:
        return np.concatenate([
            self._mean_feat(self._user_hist),
            self._mean_feat(self._sys_hist),
        ])

    # ── Reset ─────────────────────────────────────────────────────────────────

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self._user_id = random.choice(self.user_ids)
        user_rats     = self.ratings.get(self._user_id, {})

        # Histórico do usuário: últimos filmes avaliados
        rated = list(user_rats.keys())
        self._user_hist = deque(rated[-self.HIST_USER:], maxlen=self.HIST_USER)

        # Histórico do sistema: recomendações anteriores (se disponíveis)
        prev = self.rec_history.get(self._user_id, [])
        self._sys_hist = deque(prev[-self.HIST_SYS:], maxlen=self.HIST_SYS)

        self._seen   = set(user_rats.keys())
        self._step_n = 0
        return self._obs(), {}

    # ── Step ──────────────────────────────────────────────────────────────────

    def step(self, actions: dict):
        """
        Processa as ações dos 5 agentes em paralelo:
          1. Normaliza probabilidades (softmax)
          2. Seleciona um filme compatível
          3. Calcula 5 recompensas independentes
          4. Atualiza histórico
        """
        # Normaliza probabilidades
        genre_p  = _softmax(np.array(actions['genre'],    dtype=np.float32))
        gender_p = _softmax(np.array(actions['gender'],   dtype=np.float32))
        race_p   = _softmax(np.array(actions['race'],     dtype=np.float32))
        region_p = _softmax(np.array(actions['region'],   dtype=np.float32))
        lang_p   = _softmax(np.array(actions['language'], dtype=np.float32))

        # Seleciona filme
        mid = self._select_movie(genre_p, gender_p, race_p, region_p, lang_p)
        self._step_n += 1

        if mid is None:
            # Sem candidatos disponíveis
            rewards = {k: 0.0 for k in ['genre','gender','race','region','language']}
            return self._obs(), rewards, True, False, {}

        # Calcula recompensas
        user_rats = self.ratings.get(self._user_id, {})
        rewards = {
            'genre':    self._r_genre(genre_p, user_rats),
            'gender':   self._r_diversity(gender_p, 'gender_group'),
            'race':     self._r_diversity(race_p,   'race_group'),
            'region':   self._r_diversity(region_p, 'region'),
            'language': self._r_language(lang_p),
        }

        # Atualiza histórico
        self._sys_hist.appendleft(mid)
        self._seen.add(mid)

        done = self._step_n >= self.MAX_STEPS
        return self._obs(), rewards, done, False, {'movie_id': mid}

    # ── Seleção de filme ──────────────────────────────────────────────────────

    def _select_movie(self, genre_p, gender_p, race_p, region_p, lang_p):
        """Pontua filmes candidatos e amostra proporcionalmente ao score."""
        candidates = [m for m in all_movie_ids if m not in self._seen]
        if not candidates:
            return None

        # Limita candidatos para eficiência
        if len(candidates) > self.MAX_CAND:
            candidates = random.sample(candidates, self.MAX_CAND)

        scores = np.zeros(len(candidates))
        for i, mid in enumerate(candidates):
            f = self.features[mid]
            scores[i] = (
                np.dot(genre_p,  f['genres'])        +
                np.dot(gender_p, f['gender_group'])  +
                np.dot(race_p,   f['race_group'])    +
                np.dot(region_p, f['region'])        +
                np.dot(lang_p,   f['langs'])
            )

        probs = _softmax(scores * 3.0)   # temperatura baixa → mais seletivo
        return candidates[np.random.choice(len(candidates), p=probs)]

    # ── Recompensas ───────────────────────────────────────────────────────────

    def _r_genre(self, genre_p: np.ndarray, user_rats: dict) -> float:
        """
        R1 = 0.6 × preferência (rating médio ponderado por gênero)
           + 0.4 × diversidade (distância vs. gêneros dos 10 últimos filmes do usuário)
        """
        # Parte 1: rating médio ponderado por gênero
        genre_sum = np.zeros(N_GENRES)
        genre_cnt = np.zeros(N_GENRES)
        for mid, rat in user_rats.items():
            if mid in self.features:
                g = self.features[mid]['genres']
                genre_sum += g * rat
                genre_cnt += g
        avg_rat = np.where(genre_cnt > 0, genre_sum / genre_cnt, 0.0)
        pref    = float(np.dot(genre_p, avg_rat)) / 5.0   # normaliza 0-1

        # Parte 2: diversidade vs. últimos 10 filmes escolhidos pelo usuário
        recent = list(self._user_hist)[:10]
        recent_dist = np.zeros(N_GENRES)
        for mid in recent:
            if mid in self.features:
                recent_dist += self.features[mid]['genres']
        if recent_dist.sum() > 0:
            recent_dist /= recent_dist.sum()

        div = float(np.linalg.norm(genre_p - recent_dist))
        div = min(div / math.sqrt(2), 1.0)

        return 0.6 * pref + 0.4 * div

    def _r_diversity(self, probs: np.ndarray, feat_key: str) -> float:
        """
        R2, R3, R4: distância entre o vetor de probabilidades do agente
        e o centróide dos encodings dos últimos 500 filmes recomendados.
        Recompensa alta = escolha diferente do histórico recente → diversidade.
        """
        sys_list = list(self._sys_hist)
        if not sys_list:
            return 0.5

        vecs = [self.features[m][feat_key] for m in sys_list if m in self.features]
        if not vecs:
            return 0.5

        centroid = np.mean(vecs, axis=0)
        n = np.linalg.norm(centroid)
        if n > 0:
            centroid /= n

        pn = probs / (probs.sum() + 1e-8)
        return min(float(np.linalg.norm(pn - centroid)) / math.sqrt(2), 1.0)

    def _r_language(self, lang_p: np.ndarray) -> float:
        """
        R5: distância entre distribuição de idiomas do agente e centróide
        de idiomas dos últimos 500 filmes recomendados.
        """
        return self._r_diversity(lang_p, 'langs')


# Teste rápido do ambiente
env_train = MovieRecommendationEnv(movie_features, ratings, rec_history, train_users)
obs, _    = env_train.reset()
print(f'Obs shape: {obs.shape}  (esperado: ({OBS_DIM},))')

dummy_actions = {
    'genre':    np.ones(N_GENRES),
    'gender':   np.ones(3),
    'race':     np.ones(3),
    'region':   np.ones(N_REGION),
    'language': np.ones(N_LANGS),
}
obs2, rewards, done, _, info = env_train.step(dummy_actions)
print(f'Recompensas: {rewards}')
print(f'Filme recomendado: {info.get("movie_id")}')

## 8. Agentes (Actor-Critic)

Cada agente usa uma rede **Actor-Critic**:

```
obs (OBS_DIM) → [Linear 256 → ReLU] → [Linear 128 → ReLU]
                                           │                   │
                              policy_head (action_dim)   value_head (1)
                              (logits → softmax)         estimativa V(s)
```

Os **cinco agentes** rodam em paralelo sobre a mesma observação e são atualizados com suas respectivas recompensas:

| Agente | Output (logits) | Ações |
|--------|-----------------|-------|
| 1 — Gênero | `n_genres` | Top-3 gêneros com pesos [50%, 30%, 20%] |
| 2 — Dir. Gênero | 3 | Probabilidade sobre [MALE, FEMALE, MIX] |
| 3 — Dir. Raça | 3 | Probabilidade sobre [WHITE, NO_WHITE, MIX] |
| 4 — Dir. Região | `n_regions` | Probabilidade sobre 9 regiões |
| 5 — Idioma | `n_langs` | Top-3 idiomas com pesos [50%, 30%, 20%] |

In [ ]:
class AgentNet(nn.Module):
    """
    Rede Actor-Critic para um agente de recomendação.

    - Actor (policy_head): gera logits sobre as ações possíveis.
    - Critic (value_head): estima o valor do estado atual V(s).

    O treinamento usa Actor-Critic (A2C) com bônus de entropia para encorajar
    exploração no início do treino.
    """
    def __init__(self, obs_dim: int, action_dim: int, name: str = ''):
        super().__init__()
        self.name = name

        self.shared = nn.Sequential(
            nn.Linear(obs_dim, 256), nn.ReLU(),
            nn.Linear(256, 128),    nn.ReLU(),
        )
        self.policy_head = nn.Linear(128, action_dim)
        self.value_head  = nn.Linear(128, 1)

    def forward(self, obs: torch.Tensor):
        h      = self.shared(obs)
        logits = self.policy_head(h)
        value  = self.value_head(h).squeeze(-1)
        return logits, value

    def act(self, obs: torch.Tensor, temperature: float = 1.0):
        """Amostra uma ação e retorna (probs, idx, log_prob, value)."""
        logits, value = self.forward(obs)
        probs         = torch.softmax(logits / temperature, dim=-1)
        dist          = torch.distributions.Categorical(probs)
        idx           = dist.sample()
        return probs, idx, dist.log_prob(idx), value


# Cria os cinco agentes
agents = {
    'genre':    AgentNet(OBS_DIM, N_GENRES,  name='Agente 1 — Gênero'),
    'gender':   AgentNet(OBS_DIM, 3,          name='Agente 2 — Gênero Dir.'),
    'race':     AgentNet(OBS_DIM, 3,          name='Agente 3 — Raça Dir.'),
    'region':   AgentNet(OBS_DIM, N_REGION,   name='Agente 4 — Região Dir.'),
    'language': AgentNet(OBS_DIM, N_LANGS,    name='Agente 5 — Idioma'),
}

optimizers = {
    name: torch.optim.Adam(agent.parameters(), lr=3e-4)
    for name, agent in agents.items()
}

total_params = sum(p.numel() for a in agents.values() for p in a.parameters())
for name, agent in agents.items():
    n = sum(p.numel() for p in agent.parameters())
    print(f'{agent.name:<30} — {n:,} parâmetros')
print(f'{"TOTAL":<30} — {total_params:,} parâmetros')

## 9. Treinamento (Actor-Critic A2C)

### Algoritmo por agente

```
Para cada episódio:
  obs, _ = env.reset()
  Para cada passo (até MAX_STEPS):
    probs, idx, log_prob, V(s) = agente.act(obs)
    actions[agente] = probs
  obs', rewards, done = env.step(actions)
  Para cada agente:
    advantage = R - V(s)
    L_policy  = -log_prob × advantage
    L_valor   = MSE(V(s), R)
    L_entropia= -H(probs)          ← encoraja exploração
    loss      = L_policy + 0.5 × L_valor - 0.01 × L_entropia
```

> ⚠️ Reduza `N_EPISODES` para testar rapidamente (ex: 500). Para bons resultados, use ≥ 3000.

In [ ]:
def train(
    env: MovieRecommendationEnv,
    agents: dict,
    optimizers: dict,
    n_episodes: int = 3000,
    gamma: float = 0.99,
    entropy_coef: float = 0.01,
    log_every: int = 200,
) -> dict:
    """
    Treina os 5 agentes em paralelo com Actor-Critic.
    Retorna histórico de recompensas por agente.
    """
    history = {name: [] for name in agents}

    for episode in range(1, n_episodes + 1):
        obs, _ = env.reset()
        obs_t  = torch.tensor(obs, dtype=torch.float32)

        ep_rewards = {name: [] for name in agents}

        # ── Loop de passos dentro do episódio ─────────────────────────────────
        while True:
            # Todos os agentes agem simultaneamente sobre a mesma observação
            agent_data = {}
            actions    = {}

            for name, agent in agents.items():
                # Temperatura decai de 1.5 → 1.0 ao longo do treino
                temp = max(1.0, 1.5 - 0.5 * episode / n_episodes)
                probs, idx, log_prob, value = agent.act(obs_t, temperature=temp)
                agent_data[name] = (probs, log_prob, value)
                actions[name]    = probs.detach().numpy()

            # Step no ambiente
            obs_next, rewards, done, _, info = env.step(actions)
            obs_next_t = torch.tensor(obs_next, dtype=torch.float32)

            # ── Atualiza cada agente com sua recompensa ────────────────────────
            for name, agent in agents.items():
                probs, log_prob, value = agent_data[name]
                reward = float(rewards[name])
                ep_rewards[name].append(reward)

                # Bootstrap: se o episódio não terminou, usa V(s') estimado
                if not done:
                    with torch.no_grad():
                        _, v_next = agent(obs_next_t)
                    target = reward + gamma * v_next.item()
                else:
                    target = reward

                advantage   = target - value.item()
                policy_loss = -log_prob * advantage
                value_loss  = F.mse_loss(value, torch.tensor(target))
                entropy     = torch.distributions.Categorical(probs).entropy()

                loss = policy_loss + 0.5 * value_loss - entropy_coef * entropy

                optimizers[name].zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(agent.parameters(), max_norm=1.0)
                optimizers[name].step()

            obs_t = obs_next_t
            if done:
                break

        # Registra recompensa média do episódio
        for name in agents:
            history[name].append(np.mean(ep_rewards[name]))

        # Log periódico
        if episode % log_every == 0:
            means = {n: np.mean(h[-log_every:]) for n, h in history.items()}
            print(f'Ep {episode:>4} │ ' +
                  '  '.join(f'{n}={v:.3f}' for n, v in means.items()))

    return history


# ── Executa o treinamento ─────────────────────────────────────────────────────
N_EPISODES = 3000   # ← altere para 500 para teste rápido

print(f'Treinando {len(agents)} agentes por {N_EPISODES} episódios...')
history = train(env_train, agents, optimizers, n_episodes=N_EPISODES)

In [ ]:
# Visualiza curvas de aprendizado com média móvel
WINDOW = 100
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Curvas de Aprendizado — Recompensa média por episódio', fontsize=13)

colors = ['steelblue', 'darkorange', 'green', 'crimson', 'purple']
labels = ['Gênero (Ag.1)', 'Dir. Gênero (Ag.2)', 'Dir. Raça (Ag.3)',
          'Dir. Região (Ag.4)', 'Idioma (Ag.5)']

for ax, (name, rews), color, label in zip(
    axes.flat, history.items(), colors, labels
):
    ax.plot(rews, alpha=0.3, color=color)
    # Média móvel
    if len(rews) >= WINDOW:
        ma = pd.Series(rews).rolling(WINDOW).mean()
        ax.plot(ma, color=color, linewidth=2, label=f'Média {WINDOW} ep.')
    ax.set_title(label)
    ax.set_xlabel('Episódio')
    ax.set_ylabel('Recompensa')
    ax.legend()
    ax.grid(alpha=0.3)

axes.flat[-1].set_visible(False)  # oculta subplot extra
plt.tight_layout()
plt.show()

## 10. Simulação de Recomendação

Com os agentes treinados, geramos recomendações para um usuário de teste.

Para cada recomendação, o sistema exibe:
- **Top-3 gêneros** (Agente 1) com probabilidades aproximadas [50%, 30%, 20%]
- **Distribuição de gênero do diretor** (Agente 2): [MALE%, FEMALE%, MIX%]
- **Distribuição de raça do diretor** (Agente 3): [WHITE%, NO_WHITE%, MIX%]
- **Top-3 regiões do diretor** (Agente 4)
- **Top-3 idiomas** (Agente 5)

In [ ]:
def recommend(user_id: int, agents: dict, n: int = 10) -> list:
    """
    Gera n recomendações para um usuário usando os 5 agentes treinados.
    """
    for agent in agents.values():
        agent.eval()

    # Ambiente de inferência para o usuário de teste
    env_infer = MovieRecommendationEnv(
        movie_features, ratings, rec_history, [user_id]
    )
    env_infer.MAX_STEPS = n

    obs, _ = env_infer.reset()
    recs   = []

    with torch.no_grad():
        for step in range(n):
            obs_t   = torch.tensor(obs, dtype=torch.float32)
            actions = {}
            probs_display = {}

            for name, agent in agents.items():
                logits, _ = agent(obs_t)
                p = torch.softmax(logits, dim=-1).numpy()
                actions[name]       = p
                probs_display[name] = p

            obs, rewards, done, _, info = env_infer.step(actions)

            if 'movie_id' not in info:
                continue

            mid = info['movie_id']
            row = enc_df[enc_df['movieid'] == mid]
            title = row['title'].values[0] if len(row) > 0 else f'Filme {mid}'

            # Top-3 gêneros
            gp   = probs_display['genre']
            top3g = [(genre_cols[i].replace('genre_',''), f'{gp[i]:.1%}')
                     for i in np.argsort(gp)[-3:][::-1]]

            # Top-3 idiomas
            lp   = probs_display['language']
            top3l = [(lang_cols[i].replace('lang_',''), f'{lp[i]:.1%}')
                     for i in np.argsort(lp)[-3:][::-1]]

            # Top-3 regiões
            rp   = probs_display['region']
            top3r = [(REGION_NAMES[i], f'{rp[i]:.1%}')
                     for i in np.argsort(rp)[-3:][::-1]]

            recs.append({
                'step':        step + 1,
                'movie_id':    mid,
                'title':       title,
                'top3_genres': top3g,
                'top3_langs':  top3l,
                'top3_regions':top3r,
                'gender_dist': dict(zip(GENDER_GROUPS, probs_display['gender'].tolist())),
                'race_dist':   dict(zip(RACE_GROUPS,   probs_display['race'].tolist())),
                'rewards':     rewards,
            })

            if done:
                break

    return recs


# Seleciona um usuário de teste e gera recomendações
TEST_USER = test_users[0]
print(f'Gerando recomendações para usuário {TEST_USER}...')
print(f'  Filmes avaliados: {len(ratings.get(TEST_USER, {})):,}')

recs = recommend(TEST_USER, agents, n=10)

In [ ]:
# Exibe as recomendações em formato tabular
print(f'\n{"═"*90}')
print(f'  RECOMENDAÇÕES PARA O USUÁRIO {TEST_USER}')
print(f'{"═"*90}')

for r in recs:
    print(f"\n#{r['step']}  {r['title']}  (id={r['movie_id']})")
    print(f"   Gêneros  : " + '  '.join(f"{g}({p})" for g, p in r['top3_genres']))
    print(f"   Idiomas  : " + '  '.join(f"{l}({p})" for l, p in r['top3_langs']))
    print(f"   Dir. Gênero : " + '  '.join(f"{k}={v:.1%}" for k, v in r['gender_dist'].items()))
    print(f"   Dir. Raça   : " + '  '.join(f"{k}={v:.1%}" for k, v in r['race_dist'].items()))
    print(f"   Dir. Região : " + '  '.join(f"{rg}({p})" for rg, p in r['top3_regions']))
    rw = r['rewards']
    print(f"   Recompensas : gênero={rw['genre']:.3f}  gênero_dir={rw['gender']:.3f}  "
          f"raça={rw['race']:.3f}  região={rw['region']:.3f}  idioma={rw['language']:.3f}")

In [ ]:
# Visualiza diversidade das recomendações ao longo da sessão
from collections import Counter

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'Diversidade das recomendações — Usuário {TEST_USER}', fontsize=13)

# Gênero do diretor ao longo da sessão
gender_over_time = np.array(
    [[r['gender_dist'][g] for g in GENDER_GROUPS] for r in recs]
)
for i, g in enumerate(GENDER_GROUPS):
    axes[0].plot(range(1, len(recs)+1), gender_over_time[:, i], marker='o', label=g)
axes[0].set_title('Distribuição de Gênero do Diretor')
axes[0].set_xlabel('Recomendação #')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Raça do diretor ao longo da sessão
race_over_time = np.array(
    [[r['race_dist'][g] for g in RACE_GROUPS] for r in recs]
)
for i, g in enumerate(RACE_GROUPS):
    axes[1].plot(range(1, len(recs)+1), race_over_time[:, i], marker='o', label=g)
axes[1].set_title('Distribuição de Raça do Diretor')
axes[1].set_xlabel('Recomendação #')
axes[1].legend()
axes[1].grid(alpha=0.3)

# Top gêneros de filmes
genre_counter = Counter(g for r in recs for g, _ in r['top3_genres'][:1])
if genre_counter:
    keys, vals = zip(*sorted(genre_counter.items(), key=lambda x: -x[1]))
    axes[2].barh(keys, vals, color='steelblue')
    axes[2].set_title('Gêneros de Filmes Recomendados')
    axes[2].invert_yaxis()

plt.tight_layout()
plt.show()